# Forest Validation (QC) Script

This notebook checks a Beiwe study's daily summary statistics (the numbers Forest calculates from phone data) for missing, impossible, or inconsistent values. At the end it saves every problem it finds to **`validation_report.csv`**.

**How to run it:** fill in the items under "Before You Start", then click **Restart** and **Run All**. Running cells one at a time out of order can double-count issues.

## Before You Start: What You Need

**Files that must be in the same folder as this notebook:**
1. **`keyring_studies.py`**: holds your Beiwe login details. Access it here: https://github.com/onnela-lab/beiwe/blob/master/code/forest_mano/keyring_studies.py. This notebook reads 3 values from it:
   - `BEIWE_URL`: the Beiwe server address (for example `https://studies.beiwe.org`)
   - `BEIWE_ACCESS_KEY` and `BEIWE_SECRET_KEY`: your data-download API keys. To get them, log in to Beiwe, click **Manage Credentials**, and generate new Data-Download API credentials.
   - ⚠️ This file holds passwords and keys. Never commit it to git or share it.

**Things you must type in:**
1. **Study ID**: in [Setup A](#setup-a-load-the-data), set `STUDY_ID = "..."` to your study's ID.
2. **Study time zone**: in [Check 16](#check-16-raw-data-uploaded-too-late), set `STUDY_TIMEZONE` if your study isn't on US Eastern time (`"America/New_York"`).

**Other requirements:**
- Python with `pandas` and `requests` installed. Run the notebook in Jupyter or VS Code, because it uses `display()`.
- Internet access to the Beiwe server. Your API keys must have access to the study.
- Your account can download raw data. Check 17 downloads raw GPS files for a few participants, which can take several minutes. (Only row counts are saved, never locations.)

**Folders this notebook creates (safe to delete; they're just caches that speed up re-runs):** `upload_history_cache/` and `gps_summary_cache/`.

## <a id="table-of-contents"></a>Table of Contents

Click any item to jump to it.

**Setup (gets the data ready, no checking yet)**
- **[Setup A: Load the Data](#setup-a-load-the-data)**: Downloads the study's daily summary table from Beiwe so the checks below have something to look at.
- **[Setup A (alternative): Load a Local CSV Instead](#setup-a-alternative-load-a-local-csv-instead)**: Only use this if you have a downloaded CSV instead of pulling from the API. Normally skip it.
- **[Setup A (preview): Look at the Loaded Data](#setup-a-preview-look-at-the-loaded-data)**: Shows how many rows and columns were loaded and the first few rows, so you can confirm the data looks right.
- **[Setup B: Match Each Data Stream to Its Metrics](#setup-b-match-each-data-stream-to-its-metrics)**: Lists which summary numbers should come from which raw phone data (for example, GPS gives distance traveled).
- **[Setup C: Helper That Compares Raw Data to Metrics](#setup-c-helper-that-compares-raw-data-to-metrics)**: Defines a tool the later checks use. It doesn't check anything by itself.

**Checks**
- **[Check 1: Forest Processing Status](#check-1-forest-processing-status)**: Finds out which days Forest actually ran on, so we don't blame Forest for days it never processed.
- **[Check 2: Expected Columns](#check-2-expected-columns)**: Makes sure every column we expect is in the file, and flags any surprise extra columns.
- **[Check 3: Duplicate Participant-Days](#check-3-duplicate-participant-days)**: Makes sure each participant has only one row per day.
- **[Check 4: Date Validity and Calendar Gaps](#check-4-date-validity-and-calendar-gaps)**: Makes sure every date is a real date and no days are missing in the middle of a participant's data.
- **[Check 5: Numeric Data Types](#check-5-numeric-data-types)**: Makes sure columns that should hold numbers really hold numbers, not text.
- **[Check 6: No Negative Values](#check-6-no-negative-values)**: Flags any negative numbers, since counts, distances and times can't be below zero.
- **[Check 7: Duration Bounds (0 to 25 Hours)](#check-7-duration-bounds-0-to-25-hours)**: Flags daily time values (like time at home) outside 0 to 25 hours.
- **[Check 8: Observation Time Adds Up](#check-8-observation-time-adds-up)**: Checks that daytime hours + nighttime hours = total observed hours.
- **[Check 9: Call Counts vs. Call Durations](#check-9-call-counts-vs-call-durations)**: Flags days with zero calls but more than zero minutes on the phone.
- **[Check 10: Text Counts vs. Text Lengths](#check-10-text-counts-vs-text-lengths)**: Flags days with zero texts but more than zero characters texted.
- **[Check 11: Unique Contacts vs. Total Contacts](#check-11-unique-contacts-vs-total-contacts)**: Flags days where someone talked to more different people than their total number of calls or texts.
- **[Check 12: Call Reciprocity Between 0 and 1](#check-12-call-reciprocity-between-0-and-1)**: Makes sure the call reciprocity ratio stays between 0 and 1.
- **[Check 13: Cadence (Walking Pace) Makes Sense](#check-13-cadence-walking-pace-makes-sense)**: Flags walking pace that's missing on days with walking, or present on days with no walking.
- **[Check 14: GPS Movement Makes Sense](#check-14-gps-movement-makes-sense)**: Flags days with zero distance traveled but a trip distance above zero.
- **[Check 15a: Helper for Edge-of-Data Context](#check-15a-helper-for-edge-of-data-context)**: Adds notes saying whether a flagged day is the first or last day of data, or next to a gap. Used by Check 15.
- **[Check 15: Raw Data vs. Summary Metrics](#check-15-raw-data-vs-summary-metrics)**: The main check. For each day, if the phone sent raw data, are the summary metrics filled in, and the other way around?
- **[Check 16: Raw Data Uploaded Too Late?](#check-16-raw-data-uploaded-too-late)**: For days with raw data but no metrics, checks whether the data reached the server after Forest had already run.
- **[Check 17: GPS Missing-Metrics Drill-Down](#check-17-gps-missing-metrics-drill-down)**: For GPS days with no metrics at all, digs into the raw GPS to explain why (too little data, poor accuracy, or a daylight-saving bug).
- **[Final Step: Save the Report](#final-step-save-the-report)**: Combines every flagged issue from the checks above into one table and saves it as validation_report.csv.

**Reading the output:** each check prints **PASS** (nothing wrong), **FAIL** (problems found, shown in a table), or **SKIP** (couldn't run, usually because a column is missing).

## <a id="setup-a-load-the-data"></a>Setup A: Load the Data

This loads the summary statistics file from a study into a pandas dataframe and inspects the dimensions and first few rows.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
import os
import requests
import keyring_studies  # sets BEIWE_URL, BEIWE_ACCESS_KEY and BEIWE_SECRET_KEY

# >>> CHANGE THIS: paste your study's ID between the quotes. <<<
STUDY_ID = "STUDY_ID"  # enter study id here

BEIWE_SERVER = os.environ["BEIWE_URL"]
BEIWE_CREDENTIALS = {
    "access_key": os.environ["BEIWE_ACCESS_KEY"],
    "secret_key": os.environ["BEIWE_SECRET_KEY"],
}

# sends a POST request to the Beiwe server to get participant table data
response = requests.post(
    f"{BEIWE_SERVER}/get-summary-statistics/v1",
    data={**BEIWE_CREDENTIALS, "study_id": STUDY_ID}, # enter study ID here
    allow_redirects=False,
)
# if the request succeeds, save the returned data to disk
if response.status_code == 200:
    summary_statistics_table = f"summary_statistics_table_{STUDY_ID}.json"
    print(f"The data has been written to {summary_statistics_table}.")
    with open(summary_statistics_table, "wb") as f:
        f.write(response.content)
# otherwise, print the error returned by the server
else:
    print(response.content)
    print(f"There was an error downloading the data; The request returned an HTTP error of {response.status_code}")

In [ ]:
import pandas as pd
with open(summary_statistics_table, "rb") as f:
    df = pd.read_json(f)
    # this will not work unless the summary_statistics_table variable in cell above has been populated and the file exists

# Renaming columns because of different naming scheme in Forest tree vs. CSV exports.

API_COLUMN_RENAME = {
    "date": "Date",
    "participant_id": "Participant Id",
    "study_id": "Study Id",
    "timezone": "Timezone",
    "beiwe_accelerometer_bytes": "Accelerometer Bytes",
    "beiwe_app_log_bytes": "App Log Bytes",
    "beiwe_bluetooth_bytes": "Bluetooth Bytes",
    "beiwe_calls_bytes": "Calls Bytes",
    "beiwe_devicemotion_bytes": "Devicemotion Bytes",
    "beiwe_gps_bytes": "Gps Bytes",
    "beiwe_gyro_bytes": "Gyro Bytes",
    "beiwe_identifiers_bytes": "Identifiers Bytes",
    "beiwe_ios_log_bytes": "Ios Log Bytes",
    "beiwe_magnetometer_bytes": "Magnetometer Bytes",
    "beiwe_power_state_bytes": "Power State Bytes",
    "beiwe_proximity_bytes": "Proximity Bytes",
    "beiwe_reachability_bytes": "Reachability Bytes",
    "beiwe_survey_answers_bytes": "Survey Answers Bytes",
    "beiwe_survey_timings_bytes": "Survey Timings Bytes",
    "beiwe_texts_bytes": "Texts Bytes",
    "beiwe_audio_recordings_bytes": "Audio Recordings Bytes",
    "beiwe_wifi_bytes": "Wifi Bytes",
    "jasmine_distance_diameter": "Distance Diameter",
    "jasmine_distance_from_home": "Distance From Home",
    "jasmine_distance_traveled": "Distance Traveled",
    "jasmine_flight_distance_average": "Flight Distance Average",
    "jasmine_flight_distance_stddev": "Flight Distance Stddev",
    "jasmine_flight_duration_average": "Flight Duration Average",
    "jasmine_flight_duration_stddev": "Flight Duration Stddev",
    "jasmine_home_duration": "Home Duration",
    "jasmine_gyration_radius": "Gyration Radius",
    "jasmine_significant_location_count": "Significant Location Count",
    "jasmine_significant_location_entropy": "Significant Location Entropy",
    "jasmine_pause_time": "Pause Time",
    "jasmine_obs_duration": "Obs Duration",
    "jasmine_obs_day": "Obs Day",
    "jasmine_obs_night": "Obs Night",
    "jasmine_total_flight_time": "Total Flight Time",
    "jasmine_av_pause_duration": "Av Pause Duration",
    "jasmine_sd_pause_duration": "Sd Pause Duration",
    "jasmine_physical_circadian_rhythm": "Physical Circadian Rhythm",
    "jasmine_physical_circadian_rhythm_stratified": "Physical Circadian Rhythm Stratified",
    "willow_incoming_text_count": "Incoming Text Count",
    "willow_incoming_text_degree": "Incoming Text Degree",
    "willow_incoming_text_length": "Incoming Text Length",
    "willow_outgoing_text_count": "Outgoing Text Count",
    "willow_outgoing_text_degree": "Outgoing Text Degree",
    "willow_outgoing_text_length": "Outgoing Text Length",
    "willow_incoming_text_reciprocity": "Incoming Text Reciprocity",
    "willow_outgoing_text_reciprocity": "Outgoing Text Reciprocity",
    "willow_outgoing_MMS_count": "Outgoing Mms Count",
    "willow_incoming_MMS_count": "Incoming Mms Count",
    "willow_mean_responsiveness_text": "Mean Responsiveness Text",
    "willow_incoming_call_count": "Incoming Call Count",
    "willow_incoming_call_degree": "Incoming Call Degree",
    "willow_incoming_call_duration": "Incoming Call Duration",
    "willow_outgoing_call_count": "Outgoing Call Count",
    "willow_outgoing_call_degree": "Outgoing Call Degree",
    "willow_outgoing_call_duration": "Outgoing Call Duration",
    "willow_missed_call_count": "Missed Call Count",
    "willow_missed_callers": "Missed Callers",
    "willow_mean_responsiveness_call": "Mean Responsiveness Call",
    "willow_call_reciprocity": "Call Reciprocity",
    "willow_uniq_individual_call_or_text_count": "Uniq Individual Call Or Text Count",
    "oak_walking_time": "Walking Time",
    "oak_steps": "Steps",
    "oak_cadence": "Cadence",
}

df = df.rename(columns=API_COLUMN_RENAME)

## <a id="setup-a-alternative-load-a-local-csv-instead"></a>Setup A (alternative): Load a Local CSV Instead

The below chunk loads CSV file locally rather than API pull - DON'T RUN if you already loaded the data with the previous code chunks.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
import pandas as pd
# Update this path to whichever CSV you want to validate.
INPUT_CSV = "CSV_PATH" # enter local CSV path here
df = pd.read_csv(INPUT_CSV, keep_default_na=False, na_values=[""])
all_reports = []
print(df.shape)
print(df.head())


## <a id="setup-a-preview-look-at-the-loaded-data"></a>Setup A (preview): Look at the Loaded Data

This prints the size of the table (rows, columns) and its first 5 rows. It's a quick sanity check that the data loaded correctly before any checks run. If the row count is 0 or the columns look wrong, stop and fix the load step first.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
print(df.shape)
print(df.head())

## <a id="setup-b-match-each-data-stream-to-its-metrics"></a>Setup B: Match Each Data Stream to Its Metrics

This defines the mapping between each raw Beiwe data stream and the summary statistics that should be generated from that stream. The mappings are used to validate whether summary metrics are present when corresponding raw data exists.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
# Observation-time fields are excluded because they may be generated independently from the core GPS movement metrics.
all_reports = []

STREAMS = {
    "GPS": {
        "raw_cols": ["Gps Bytes"],
        "metric_cols": [
            "Distance Diameter",
            "Distance From Home",
            "Distance Traveled",
            "Flight Distance Average",
            "Flight Distance Stddev",
            "Flight Duration Average",
            "Flight Duration Stddev",
            "Home Duration",
            "Gyration Radius",
            "Significant Location Count",
            "Significant Location Entropy",
            "Total Flight Time",
        ],
    },
    "Accelerometer": {
        "raw_cols": ["Accelerometer Bytes"],
        # Cadence is excluded here on purpose: Forest (oak/runners.py) sets
        # it to NaN whenever there were zero walking bouts that day, which
        # is expected and correct, not a processing gap. It gets its own
        # dedicated Cadence Validity Check below instead.
        "metric_cols": ["Walking Time", "Steps"],
    },
    "Calls+Texts": {
        # Willow processes calls and texts together: it runs if EITHER
        # stream has raw data that day, and fills in real zeros (not
        # blanks) for whichever stream was quiet. So this has to be one
        # combined check, not two independent per-stream checks -- otherwise
        # a normal "Willow ran because of calls, correctly reported 0 texts"
        # day gets misflagged as a processing gap.
        "raw_cols": ["Calls Bytes", "Texts Bytes"],
        "metric_cols": [
            "Incoming Call Count",
            "Incoming Call Degree",
            "Incoming Call Duration",
            "Outgoing Call Count",
            "Outgoing Call Degree",
            "Outgoing Call Duration",
            "Missed Call Count",
            "Missed Callers",
            "Incoming Text Count",
            "Incoming Text Degree",
            "Incoming Text Length",
            "Outgoing Text Count",
            "Outgoing Text Degree",
            "Outgoing Text Length",
            "Incoming Text Reciprocity",
            "Outgoing Text Reciprocity",
            "Outgoing Mms Count",
            "Incoming Mms Count",
        ],
    },
}

## <a id="check-1-forest-processing-status"></a>Check 1: Forest Processing Status

Forest metrics only exist for participant-days Forest actually ran on, so a day Forest never processed shouldn't be judged as a mismatch. This pulls every Forest task the server has run for the study (the `get-forest-tasks/v1` endpoint) and builds a lookup of (participant, tree, day) -> what happened. Each successful run overwrites the summary values for the days it outputs, and a failed run never erases them, so a day is "Processed" if any successful run with output covered it. The Raw Data vs. Summary Metrics check below only validates "Processed" days and reports the rest separately.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
import os
import requests
import keyring_studies  # sets BEIWE_URL, BEIWE_ACCESS_KEY and BEIWE_SECRET_KEY

BEIWE_SERVER = os.environ["BEIWE_URL"]
BEIWE_CREDENTIALS = {
    "access_key": os.environ["BEIWE_ACCESS_KEY"],
    "secret_key": os.environ["BEIWE_SECRET_KEY"],
}

TREE_FOR_STREAM = {"GPS": "jasmine", "Accelerometer": "oak", "Calls+Texts": "willow"}

# Works for either load path (API pull or local CSV).
validation_study_id = df["Study Id"].dropna().iloc[0]

response = requests.post(
    f"{BEIWE_SERVER}/get-forest-tasks/v1",
    data={**BEIWE_CREDENTIALS, "study_id": validation_study_id},
    allow_redirects=False,
)
response.raise_for_status()
# The server pages through tasks 1000 at a time internally but streams them back
# as one JSON list, so a single request returns everything.
forest_tasks = pd.DataFrame(response.json())

print("===== Forest Tasks =====")
print(f"{len(forest_tasks)} Forest task(s) found for study {validation_study_id}")
display(pd.crosstab(forest_tasks["forest_tree"], forest_tasks["status"]))

# WHY: Forest can run on the same day more than once (e.g. one run fails, a later
# one succeeds). We keep the BEST outcome per day, because a failed run never
# erases numbers an earlier successful run already wrote.
FOREST_STATUS_RANK = {"Processed": 3, "Ran, no output": 2, "Failed": 1}

coverage_rows = []
for task in forest_tasks.itertuples():
    if task.status == "success":
        forest_status = "Processed" if task.forest_output_exists else "Ran, no output"
    elif task.status in ("error", "cancelled"):
        forest_status = "Failed"
    else:  # queued or running: nothing written yet
        continue

    forest_error = (
        task.stacktrace.strip().split("\n")[-1]
        if forest_status == "Failed" and isinstance(task.stacktrace, str)
        else pd.NA
    )

    # data_date_start and data_date_end are both inclusive
    for day in pd.date_range(task.data_date_start, task.data_date_end, freq="D"):
        coverage_rows.append(
            (
                task.patient_id, task.forest_tree, day, forest_status, forest_error,
                task.process_start_time, task.data_date_start, task.data_date_end,
            )
        )

forest_coverage = pd.DataFrame(
    coverage_rows,
    columns=[
        "Participant Id", "Forest Tree", "Date", "Forest Status", "Forest Error",
        "Forest Run Started", "Forest Data Start", "Forest Data End",
    ],
)

# Keep the best outcome per day; among successful runs, keep the most recent one.
forest_coverage["rank"] = forest_coverage["Forest Status"].map(FOREST_STATUS_RANK)
forest_coverage = (
    forest_coverage.sort_values(["rank", "Forest Run Started"])
    .groupby(["Participant Id", "Forest Tree", "Date"])
    .tail(1)
    .drop(columns="rank")
)


def add_forest_status(report, stream_name):
    lookup = (
        forest_coverage[forest_coverage["Forest Tree"] == TREE_FOR_STREAM[stream_name]]
        .drop(columns="Forest Tree")
        .rename(columns={"Date": "_day"})
    )
    report = report.assign(_day=pd.to_datetime(report["Date"]))
    report = report.merge(lookup, on=["Participant Id", "_day"], how="left").drop(columns="_day")
    report["Forest Status"] = report["Forest Status"].fillna("Never processed")
    return report

## <a id="setup-c-helper-that-compares-raw-data-to-metrics"></a>Setup C: Helper That Compares Raw Data to Metrics

This creates a validation function that compares raw data availability against summary statistic availability for a given data stream. Specifically, this function identifies days with raw data present (raw_col > 0) and checks if all expected summary metrics are available. Then, it returns a report containing the participant ID, date, stream name, and issue type if there are any flags.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
def check_stream(df, stream_name, raw_cols, metric_cols):
    # raw_present is True if ANY of the raw columns has data that day --
    # e.g. Willow runs if calls OR texts (or both) have raw bytes, so Calls
    # and Texts must be checked together, not as two independent streams.
    raw_present = pd.concat(
        [df[col].fillna(0) > 0 for col in raw_cols], axis=1
    ).any(axis=1)
# A metric of 0 is considered present. Only a blank or NaN value is considered missing. 
    missing_metric_count = df[metric_cols].isna().sum(axis=1)
    all_metrics_missing = missing_metric_count == len(metric_cols)
    some_metrics_missing = (missing_metric_count > 0) & ~all_metrics_missing
    metrics_present = missing_metric_count == 0

    # A day where every metric is exactly 0 with no raw data for this stream is
    # normally a real "nothing happened" day -- e.g. the call/text loggers upload
    # header-only files when there were no calls or texts, which show up as no
    # bytes, and Willow correctly reports all zeros. But Willow also writes zeros
    # for days the phone was off (it fills every day between the first and last
    # event), so all-zero metrics only count as real if the phone uploaded
    # something that day.
    all_metrics_zero = (df[metric_cols].fillna(0) == 0).all(axis=1)
    phone_active = (df.filter(like=" Bytes").fillna(0) > 0).any(axis=1)

    issue = pd.Series([None] * len(df), index=df.index)

    issue[raw_present & all_metrics_missing] = "Raw data present but ALL metrics missing"
    issue[raw_present & some_metrics_missing] = "Raw data present but SOME metrics missing"
    issue[~raw_present & metrics_present & ~all_metrics_zero] = "Metrics present but raw data missing"
    issue[~raw_present & metrics_present & all_metrics_zero & ~phone_active] = (
        "Metrics are all zero but the phone sent no data that day"
    )

    report = df.loc[issue.notna(), ["Participant Id", "Date"]].copy()
    report["Stream"] = stream_name
    report["Issue"] = issue[issue.notna()].values

    return report

## <a id="check-2-expected-columns"></a>Check 2: Expected Columns

This ensures that the expected columns are present in the data.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
expected_columns = [
    "Date",
    "Participant Id",
    "Study Id",
    "Timezone",
    "Accelerometer Bytes",
    "App Log Bytes",
    "Bluetooth Bytes",
    "Calls Bytes",
    "Devicemotion Bytes",
    "Gps Bytes",
    "Gyro Bytes",
    "Identifiers Bytes",
    "Ios Log Bytes",
    "Magnetometer Bytes",
    "Power State Bytes",
    "Proximity Bytes",
    "Reachability Bytes",
    "Survey Answers Bytes",
    "Survey Timings Bytes",
    "Texts Bytes",
    "Audio Recordings Bytes",
    "Wifi Bytes",
    "Distance Diameter",
    "Distance From Home",
    "Distance Traveled",
    "Flight Distance Average",
    "Flight Distance Stddev",
    "Flight Duration Average",
    "Flight Duration Stddev",
    "Home Duration",
    "Gyration Radius",
    "Significant Location Count",
    "Significant Location Entropy",
    "Pause Time",
    "Obs Duration",
    "Obs Day",
    "Obs Night",
    "Total Flight Time",
    "Av Pause Duration",
    "Sd Pause Duration",
    "Physical Circadian Rhythm",
    "Physical Circadian Rhythm Stratified",
    "Incoming Text Count",
    "Incoming Text Degree",
    "Incoming Text Length",
    "Outgoing Text Count",
    "Outgoing Text Degree",
    "Outgoing Text Length",
    "Incoming Text Reciprocity",
    "Outgoing Text Reciprocity",
    "Outgoing Mms Count",
    "Incoming Mms Count",
    "Mean Responsiveness Text",
    "Incoming Call Count",
    "Incoming Call Degree",
    "Incoming Call Duration",
    "Outgoing Call Count",
    "Outgoing Call Degree",
    "Outgoing Call Duration",
    "Missed Call Count",
    "Missed Callers",
    "Mean Responsiveness Call",
    "Call Reciprocity",
    "Uniq Individual Call Or Text Count",
    "Walking Time",
    "Steps",
    "Cadence"
]

# WHY: every later check looks up columns by name. If a column is missing or
# was renamed, those checks would silently skip it, so we catch it up front.
missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

extra_columns = [
    col for col in df.columns
    if col not in expected_columns
]

print("===== Column Schema Check =====")

schema_issues = []

for col in missing_columns:
    schema_issues.append({
        "Participant Id": pd.NA,
        "Date": pd.NA,
        "Check": "Expected Columns",
        "Stream": pd.NA,
        "Column": col,
        "Issue": "Expected column is missing"
    })

for col in extra_columns:
    schema_issues.append({
        "Participant Id": pd.NA,
        "Date": pd.NA,
        "Check": "Expected Columns",
        "Stream": pd.NA,
        "Column": col,
        "Issue": "Unexpected column is present"
    })

if not schema_issues:
    print("PASS: All expected columns are present.")
else:
    schema_report = pd.DataFrame(schema_issues)
    all_reports.append(schema_report)

    print(f"FAIL: {len(schema_report)} schema issue(s) found.")
    display(schema_report)

## <a id="check-3-duplicate-participant-days"></a>Check 3: Duplicate Participant-Days

This checks whether the same Participant Id + Date combination appears more than once in the export. A duplicate would cause every other row-level check below to count that row's issues twice, so this runs before any of them.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
print("===== Duplicate Participant-Days Check =====")

# WHY: each participant should have exactly one row per day. A duplicate row
# would make every check below report the same problem twice.
duplicate_counts = (
    df.groupby(["Participant Id", "Date"])
    .size()
    .reset_index(name="Count")
)
duplicate_counts = duplicate_counts[duplicate_counts["Count"] > 1]

if duplicate_counts.empty:
    print("PASS: No duplicate participant-day rows found.")
else:
    duplicate_report = duplicate_counts[["Participant Id", "Date"]].copy()
    duplicate_report["Check"] = "Duplicate Participant-Days"
    duplicate_report["Stream"] = pd.NA
    duplicate_report["Column"] = pd.NA
    duplicate_report["Issue"] = (
        "Appears " + duplicate_counts["Count"].astype(str) + " times in the export"
    )

    all_reports.append(duplicate_report)

    print(
        f"FAIL: {len(duplicate_report)} participant-day combination(s) "
        "are duplicated in the export."
    )
    display(duplicate_report)

## <a id="check-4-date-validity-and-calendar-gaps"></a>Check 4: Date Validity and Calendar Gaps

This checks two things. First, whether every Date value can actually be parsed as a real date. Second, for each participant, whether every calendar day between their earliest and latest row is present - a missing day in the middle would mean a row silently dropped out of the export.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
print("===== Date Validity Check =====")

parsed_dates = pd.to_datetime(df["Date"], errors="coerce")
unparseable_mask = parsed_dates.isna() & df["Date"].notna()

if not unparseable_mask.any():
    print("PASS: All dates parsed successfully.")
else:
    unparseable_report = df.loc[unparseable_mask, ["Participant Id", "Date"]].copy()
    unparseable_report["Check"] = "Date Validity"
    unparseable_report["Stream"] = pd.NA
    unparseable_report["Column"] = "Date"
    unparseable_report["Issue"] = "Date could not be parsed"

    all_reports.append(unparseable_report)

    print(f"FAIL: {len(unparseable_report)} row(s) have an unparseable Date.")
    display(unparseable_report)

print("\n===== Calendar Gap Check =====")

# Only look at rows with a parseable date -- an unparseable one is already
# caught above and would break the day-range math below.
dated = df.loc[parsed_dates.notna()].copy()
dated["Parsed Date"] = parsed_dates.loc[parsed_dates.notna()]

# WHY: the export should have a row for every day between a participant's first
# and last day, even if the phone sent nothing that day. A missing day in the
# middle means a row was dropped somewhere, not that the phone was off.
gap_reports = []

for participant_id, group in dated.groupby("Participant Id"):
    observed_days = pd.to_datetime(group["Parsed Date"].unique())
    full_range = pd.date_range(observed_days.min(), observed_days.max(), freq="D")
    missing_days = full_range.difference(observed_days)

    if len(missing_days) > 0:
        gap_reports.append(pd.DataFrame({
            "Participant Id": participant_id,
            "Date": missing_days,
            "Check": "Calendar Gap",
            "Stream": pd.NA,
            "Column": pd.NA,
            "Issue": "No row exists for this date within the participant's collection range"
        }))

if not gap_reports:
    print("PASS: No calendar gaps found within any participant's date range.")
else:
    gap_report = pd.concat(gap_reports, ignore_index=True)
    all_reports.append(gap_report)

    print(f"FAIL: {len(gap_report)} missing calendar day(s) found across participants.")
    display(gap_report)

## <a id="check-5-numeric-data-types"></a>Check 5: Numeric Data Types

This ensures that columns that are expected to have numeric data are indeed numeric.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
# Build the numeric column list from the defined stream mappings.

stream_numeric_columns = []

for config in STREAMS.values():
    stream_numeric_columns.extend(config["raw_cols"])
    stream_numeric_columns.extend(config["metric_cols"])

additional_numeric_columns = [
    "Pause Time",
    "Obs Duration",
    "Obs Day",
    "Obs Night",
    "Av Pause Duration",
    "Sd Pause Duration",
    "Physical Circadian Rhythm",
    "Physical Circadian Rhythm Stratified",
    "Mean Responsiveness Text",
    "Mean Responsiveness Call",
    "Call Reciprocity",
    "Uniq Individual Call Or Text Count"
]

numeric_columns = list(
    dict.fromkeys(
        stream_numeric_columns + additional_numeric_columns
    )
)

numeric_columns = [
    col for col in numeric_columns
    if col in df.columns
]

# WHY: if a number column was read in as text (e.g. because of a stray word
# like "NA"), math-based checks below like "> 0" or "< 25" won't work properly on it.
incorrect_types = []

for col in numeric_columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        incorrect_types.append({
            "Participant Id": pd.NA,
            "Date": pd.NA,
            "Check": "Data Type Validation",
            "Stream": pd.NA,
            "Column": col,
            "Issue": f"Expected numeric data type but found {df[col].dtype}"
        })

print("===== Data Type Validation =====")

if not incorrect_types:
    print("PASS: All expected numeric columns have numeric data types.")
else:
    data_type_report = pd.DataFrame(incorrect_types)
    all_reports.append(data_type_report)

    print(
        f"FAIL: {len(data_type_report)} column(s) "
        "have incorrect data types."
    )
    display(data_type_report)

## <a id="check-6-no-negative-values"></a>Check 6: No Negative Values

This ensures that there are no negative values in numeric columns.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
# WHY: everything checked here is a count, a distance, a duration, or a byte
# size. None of those can be negative, so any negative value is an error.
negative_reports = []

print("===== Range Checks: Non-negative Values =====")

for col in numeric_columns:
    values = pd.to_numeric(df[col], errors="coerce")
    negative_mask = values.notna() & (values < 0)

    if negative_mask.any():
        invalid_rows = df.loc[
            negative_mask,
            ["Participant Id", "Date", col]
        ].copy()

        invalid_rows["Check"] = "Range Check"
        invalid_rows["Stream"] = pd.NA
        invalid_rows["Column"] = col
        invalid_rows["Issue"] = (
            f"Negative value found: "
            + invalid_rows[col].astype(str)
        )

        negative_reports.append(
            invalid_rows[
                [
                    "Participant Id",
                    "Date",
                    "Check",
                    "Stream",
                    "Column",
                    "Issue"
                ]
            ]
        )

if not negative_reports:
    print("PASS: No negative values found in numeric metric columns.")
else:
    negative_values_report = pd.concat(
        negative_reports,
        ignore_index=True
    )

    all_reports.append(negative_values_report)

    print(
        f"FAIL: {len(negative_values_report)} "
        "negative value issue(s) found."
    )
    display(negative_values_report)

## <a id="check-7-duration-bounds-0-to-25-hours"></a>Check 7: Duration Bounds (0 to 25 Hours)

This check identifies duration variables that fall outside their natural range. Daily duration variables must be between 0 and 25 hours.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
# WHY 0 to 25 hours (and not 0 to 24)?
# A day has 24 hours, but these values can legitimately go a little over 24:
#   - Forest fills in (imputes) missing GPS time, and the filled-in time can
#     spill slightly past 24 hours.
#   - The day the clocks "fall back" for daylight saving time really is 25 hours long.
# Using 25 as the upper limit allows for that, so we only flag values that are
# truly impossible (below 0, or more than 25 hours in one day).
duration_bounds = {
    "Home Duration": (0, 25),
    "Obs Day": (0, 25),
    "Obs Night": (0, 25),
    "Obs Duration": (0, 25),
}

physical_bound_reports = []

print("===== Physical Bounds Check =====")

for col, (lower_bound, upper_bound) in duration_bounds.items():
    if col not in df.columns:
        continue

    values = pd.to_numeric(df[col], errors="coerce")

    invalid_mask = (
        values.notna()
        & (
            (values < lower_bound)
            | (values > upper_bound)
        )
    )

    if invalid_mask.any():
        invalid_rows = df.loc[
            invalid_mask,
            ["Participant Id", "Date", col]
        ].copy()

        invalid_rows["Check"] = "Physical Bounds"
        invalid_rows["Stream"] = pd.NA
        invalid_rows["Column"] = col
        invalid_rows["Issue"] = (
            f"{col} must be between "
            f"{lower_bound} and {upper_bound} hours"
        )

        physical_bound_reports.append(
            invalid_rows[
                [
                    "Participant Id",
                    "Date",
                    "Check",
                    "Stream",
                    "Column",
                    "Issue"
                ]
            ]
        )

if not physical_bound_reports:
    print("PASS: All duration variables are within physical bounds.")
else:
    physical_bounds_report = pd.concat(
        physical_bound_reports,
        ignore_index=True
    )

    all_reports.append(physical_bounds_report)

    print(
        f"FAIL: {len(physical_bounds_report)} "
        "physical-bound issue(s) found."
    )
    display(physical_bounds_report)

## <a id="check-8-observation-time-adds-up"></a>Check 8: Observation Time Adds Up

This checks that observation day + observation night = observation duration (approximately).

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
required_cols = [
    "Participant Id",
    "Date",
    "Obs Day",
    "Obs Night",
    "Obs Duration"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

print("===== Observation Duration Consistency Check =====")

if missing_cols:
    print(f"SKIP: Missing required columns: {missing_cols}")
else:
    # WHY a tolerance? Day + Night should equal Total, but decimals can round
    # slightly differently (e.g. 23.999999 vs 24.0). Allowing a 0.01-hour
    # (about 36 seconds) difference avoids flagging rounding noise.
    tolerance = 0.01

    invalid_mask = (
        df["Obs Day"].notna()
        & df["Obs Night"].notna()
        & df["Obs Duration"].notna()
        & (
            (
                df["Obs Day"]
                + df["Obs Night"]
                - df["Obs Duration"]
            ).abs() > tolerance
        )
    )

    if not invalid_mask.any():
        print(
            "PASS: Obs Day + Obs Night is approximately "
            "equal to Obs Duration."
        )
    else:
        observation_report = df.loc[
            invalid_mask,
            [
                "Participant Id",
                "Date",
                "Obs Day",
                "Obs Night",
                "Obs Duration"
            ]
        ].copy()

        observation_report["Check"] = (
            "Observation Duration Consistency"
        )
        observation_report["Stream"] = "GPS"
        observation_report["Column"] = "Obs Duration"
        observation_report["Issue"] = (
            "Obs Day + Obs Night does not equal Obs Duration"
        )

        observation_report = observation_report[
            [
                "Participant Id",
                "Date",
                "Check",
                "Stream",
                "Column",
                "Issue"
            ]
        ]

        all_reports.append(observation_report)

        print(
            f"FAIL: {len(observation_report)} "
            "observation-duration issue(s) found."
        )
        display(observation_report)

## <a id="check-9-call-counts-vs-call-durations"></a>Check 9: Call Counts vs. Call Durations

This chunk will flag if the incoming call count = 0 and incoming call duration is > 0 because you cannot accumulate call duration with no calls.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
required_cols = [
    "Participant Id",
    "Date",
    "Incoming Call Count",
    "Incoming Call Duration",
    "Outgoing Call Count",
    "Outgoing Call Duration"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

print("===== Call Counts vs. Durations Check =====")

# WHY: call duration is the total time spent on calls that day. If there were
# zero calls, there can't be any call time, so a duration above 0 is impossible.
call_reports = []

if missing_cols:
    print(f"SKIP: Missing required columns: {missing_cols}")
else:
    incoming_mask = (
        (df["Incoming Call Count"] == 0)
        & (df["Incoming Call Duration"] > 0)
    )

    if incoming_mask.any():
        incoming_report = df.loc[
            incoming_mask,
            ["Participant Id", "Date"]
        ].copy()

        incoming_report["Check"] = "Call Counts vs. Durations"
        incoming_report["Stream"] = "Calls"
        incoming_report["Column"] = "Incoming Call Duration"
        incoming_report["Issue"] = (
            "Incoming Call Count is 0 but "
            "Incoming Call Duration is greater than 0"
        )

        call_reports.append(incoming_report)

    outgoing_mask = (
        (df["Outgoing Call Count"] == 0)
        & (df["Outgoing Call Duration"] > 0)
    )

    if outgoing_mask.any():
        outgoing_report = df.loc[
            outgoing_mask,
            ["Participant Id", "Date"]
        ].copy()

        outgoing_report["Check"] = "Call Counts vs. Durations"
        outgoing_report["Stream"] = "Calls"
        outgoing_report["Column"] = "Outgoing Call Duration"
        outgoing_report["Issue"] = (
            "Outgoing Call Count is 0 but "
            "Outgoing Call Duration is greater than 0"
        )

        call_reports.append(outgoing_report)

    if not call_reports:
        print("PASS: No call count/duration inconsistencies found.")
    else:
        call_consistency_report = pd.concat(
            call_reports,
            ignore_index=True
        )

        all_reports.append(call_consistency_report)

        print(
            f"FAIL: {len(call_consistency_report)} "
            "call consistency issue(s) found."
        )
        display(call_consistency_report)

## <a id="check-10-text-counts-vs-text-lengths"></a>Check 10: Text Counts vs. Text Lengths

This will flag if the incoming or outgoing text count = 0 and the corresponding text length is > 0.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
required_cols = [
    "Participant Id",
    "Date",
    "Incoming Text Count",
    "Incoming Text Length",
    "Outgoing Text Count",
    "Outgoing Text Length"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

print("===== Text Counts vs. Lengths Check =====")

# WHY: text length is the total characters across that day's texts. If there
# were zero texts, there can't be any characters, so a length above 0 is impossible.
text_reports = []

if missing_cols:
    print(f"SKIP: Missing required columns: {missing_cols}")
else:
    incoming_mask = (
        (df["Incoming Text Count"] == 0)
        & (df["Incoming Text Length"] > 0)
    )

    if incoming_mask.any():
        incoming_report = df.loc[
            incoming_mask,
            ["Participant Id", "Date"]
        ].copy()

        incoming_report["Check"] = "Text Counts vs. Lengths"
        incoming_report["Stream"] = "Texts"
        incoming_report["Column"] = "Incoming Text Length"
        incoming_report["Issue"] = (
            "Incoming Text Count is 0 but "
            "Incoming Text Length is greater than 0"
        )

        text_reports.append(incoming_report)

    outgoing_mask = (
        (df["Outgoing Text Count"] == 0)
        & (df["Outgoing Text Length"] > 0)
    )

    if outgoing_mask.any():
        outgoing_report = df.loc[
            outgoing_mask,
            ["Participant Id", "Date"]
        ].copy()

        outgoing_report["Check"] = "Text Counts vs. Lengths"
        outgoing_report["Stream"] = "Texts"
        outgoing_report["Column"] = "Outgoing Text Length"
        outgoing_report["Issue"] = (
            "Outgoing Text Count is 0 but "
            "Outgoing Text Length is greater than 0"
        )

        text_reports.append(outgoing_report)

    if not text_reports:
        print("PASS: No text count/length inconsistencies found.")
    else:
        text_consistency_report = pd.concat(
            text_reports,
            ignore_index=True
        )

        all_reports.append(text_consistency_report)

        print(
            f"FAIL: {len(text_consistency_report)} "
            "text consistency issue(s) found."
        )
        display(text_consistency_report)

## <a id="check-11-unique-contacts-vs-total-contacts"></a>Check 11: Unique Contacts vs. Total Contacts

"Degree" is the number of unique people someone called/texted, and "Count" is the total number of calls/texts. Degree can never be higher than Count, since we can't have more unique contacts than total interactions.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
print("===== Degree vs. Count Check =====")

# WHY: "Degree" = number of DIFFERENT people; "Count" = TOTAL calls/texts.
# Calling 2 people 5 times gives Degree 2, Count 5. You can't have more unique
# people than total calls, so Degree > Count is impossible.
degree_count_pairs = [
    ("Incoming Call Degree", "Incoming Call Count", "Calls"),
    ("Outgoing Call Degree", "Outgoing Call Count", "Calls"),
    ("Incoming Text Degree", "Incoming Text Count", "Texts"),
    ("Outgoing Text Degree", "Outgoing Text Count", "Texts"),
]

degree_reports = []

for degree_col, count_col, stream_name in degree_count_pairs:
    if degree_col not in df.columns or count_col not in df.columns:
        continue

    degree_values = pd.to_numeric(df[degree_col], errors="coerce")
    count_values = pd.to_numeric(df[count_col], errors="coerce")

    invalid_mask = (
        degree_values.notna()
        & count_values.notna()
        & (degree_values > count_values)
    )

    if invalid_mask.any():
        invalid_rows = df.loc[
            invalid_mask,
            ["Participant Id", "Date", degree_col, count_col]
        ].copy()

        invalid_rows["Check"] = "Degree vs. Count"
        invalid_rows["Stream"] = stream_name
        invalid_rows["Column"] = degree_col
        invalid_rows["Issue"] = f"{degree_col} is greater than {count_col}"

        degree_reports.append(
            invalid_rows[
                [
                    "Participant Id",
                    "Date",
                    "Check",
                    "Stream",
                    "Column",
                    "Issue"
                ]
            ]
        )

if not degree_reports:
    print("PASS: No degree-versus-count inconsistencies found.")
else:
    degree_report = pd.concat(degree_reports, ignore_index=True)
    all_reports.append(degree_report)

    print(
        f"FAIL: {len(degree_report)} "
        "degree-versus-count issue(s) found."
    )
    display(degree_report)

## <a id="check-12-call-reciprocity-between-0-and-1"></a>Check 12: Call Reciprocity Between 0 and 1

"Call Reciprocity" is a true ratio (1 = perfect reciprocity, 0 = one-directional), so it should always fall between 0 and 1. Note: despite the similar name, "Incoming/Outgoing Text Reciprocity" are NOT ratios - Forest defines them as raw counts ("number of received/sent SMS without response"), so they're intentionally excluded here. They are already covered by the Non-Negative Values check above, which is the only real constraint on a count.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
# WHY: Call Reciprocity is a ratio (a fraction), so it must be between 0 and 1.
# Text Reciprocity is left out on purpose: despite the name, Forest stores it as
# a count, not a ratio (see the explanation above this cell).
reciprocity_bounds = {
    "Call Reciprocity": (0, 1),
}

reciprocity_reports = []

print("===== Reciprocity Bounds Check =====")

for col, (lower_bound, upper_bound) in reciprocity_bounds.items():
    if col not in df.columns:
        continue

    values = pd.to_numeric(df[col], errors="coerce")

    invalid_mask = (
        values.notna()
        & (
            (values < lower_bound)
            | (values > upper_bound)
        )
    )

    if invalid_mask.any():
        invalid_rows = df.loc[
            invalid_mask,
            ["Participant Id", "Date", col]
        ].copy()

        invalid_rows["Check"] = "Reciprocity Bounds"
        invalid_rows["Stream"] = pd.NA
        invalid_rows["Column"] = col
        invalid_rows["Issue"] = (
            f"{col} must be between {lower_bound} and {upper_bound}"
        )

        reciprocity_reports.append(
            invalid_rows[
                [
                    "Participant Id",
                    "Date",
                    "Check",
                    "Stream",
                    "Column",
                    "Issue"
                ]
            ]
        )

if not reciprocity_reports:
    print("PASS: All reciprocity values are within 0 and 1.")
else:
    reciprocity_report = pd.concat(
        reciprocity_reports,
        ignore_index=True
    )

    all_reports.append(reciprocity_report)

    print(
        f"FAIL: {len(reciprocity_report)} "
        "reciprocity-bound issue(s) found."
    )
    display(reciprocity_report)

## <a id="check-13-cadence-walking-pace-makes-sense"></a>Check 13: Cadence (Walking Pace) Makes Sense

Forest sets Cadence to NaN whenever there were zero walking bouts that day, which is expected and correct - not a processing gap. The only genuine problems are Cadence missing despite real walking time, or Cadence present despite zero walking time.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
required_cols = ["Participant Id", "Date", "Walking Time", "Cadence"]
missing_cols = [col for col in required_cols if col not in df.columns]

print("===== Cadence Validity Check =====")

if missing_cols:
    print(f"SKIP: Missing required columns: {missing_cols}")
else:
    walking_time = pd.to_numeric(df["Walking Time"], errors="coerce")
    cadence_missing = df["Cadence"].isna()

    # WHY: cadence = steps per minute WHILE walking. With no walking, there is
    # nothing to divide by, so a blank cadence is correct. It's only a problem if
    # cadence is blank despite walking, or filled in despite no walking.
    unexpected_missing_mask = (walking_time > 0) & cadence_missing
    unexpected_present_mask = (walking_time == 0) & ~cadence_missing

    cadence_reports = []

    if unexpected_missing_mask.any():
        r = df.loc[unexpected_missing_mask, ["Participant Id", "Date"]].copy()
        r["Check"] = "Cadence Validity"
        r["Stream"] = "Accelerometer"
        r["Column"] = "Cadence"
        r["Issue"] = "Cadence is missing despite Walking Time greater than 0"
        cadence_reports.append(r)

    if unexpected_present_mask.any():
        r = df.loc[unexpected_present_mask, ["Participant Id", "Date"]].copy()
        r["Check"] = "Cadence Validity"
        r["Stream"] = "Accelerometer"
        r["Column"] = "Cadence"
        r["Issue"] = "Cadence is present despite Walking Time equal to 0"
        cadence_reports.append(r)

    if not cadence_reports:
        print("PASS: Cadence is blank only on days with zero walking time, as expected.")
    else:
        cadence_report = pd.concat(cadence_reports, ignore_index=True)
        all_reports.append(cadence_report)

        print(f"FAIL: {len(cadence_report)} cadence consistency issue(s) found.")
        display(cadence_report)

## <a id="check-14-gps-movement-makes-sense"></a>Check 14: GPS Movement Makes Sense

This will flag if distance traveled = 0 and flight distance average is > 0.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
required_cols = [
    "Participant Id",
    "Date",
    "Distance Traveled",
    "Flight Distance Average"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

print("===== GPS Movement Consistency Check =====")

if missing_cols:
    print(f"SKIP: Missing required columns: {missing_cols}")
else:
    # WHY: "Distance Traveled" is the total distance for the day, and "Flight
    # Distance Average" is the average length of each trip. If total distance
    # is 0, no trip could have covered any distance.
    invalid_mask = (
        (df["Distance Traveled"] == 0)
        & (df["Flight Distance Average"] > 0)
    )

    if not invalid_mask.any():
        print("PASS: No GPS movement inconsistencies found.")
    else:
        gps_report = df.loc[
            invalid_mask,
            ["Participant Id", "Date"]
        ].copy()

        gps_report["Check"] = "GPS Movement Consistency"
        gps_report["Stream"] = "GPS"
        gps_report["Column"] = "Flight Distance Average"
        gps_report["Issue"] = (
            "Distance Traveled is 0 but "
            "Flight Distance Average is greater than 0"
        )

        all_reports.append(gps_report)

        print(
            f"FAIL: {len(gps_report)} "
            "GPS movement issue(s) found."
        )
        display(gps_report)

## <a id="check-15a-helper-for-edge-of-data-context"></a>Check 15a: Helper for Edge-of-Data Context

For a flagged stream's report, this adds two context columns: "First/Last Day?" (is this that stream's first or last day of actual raw data for this participant?) and "Next to Gap?" (is the day immediately before or after this one missing raw data for this stream entirely?). Strange values at the edges of collection or next to a gap may be expected rather than genuine bugs, so this gives more context for interpreting a flag. This is only used by the Raw Data vs. Summary Metrics check below.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
def add_edge_context(df, report, raw_cols):
    if report.empty:
        report["First/Last Day?"] = pd.Series(dtype=bool)
        report["Next to Gap?"] = pd.Series(dtype=bool)
        return report

    # "Active" = this stream actually had raw bytes that day (not just a
    # padded row in the export).
    raw_present = pd.concat(
        [df[col].fillna(0) > 0 for col in raw_cols], axis=1
    ).any(axis=1)

    active = df.loc[raw_present, ["Participant Id", "Date"]].copy()
    active["Date"] = pd.to_datetime(active["Date"])

    first_last = active.groupby("Participant Id")["Date"].agg(["min", "max"])
    first_last.columns = ["First Day", "Last Day"]

    # For each participant, the calendar days between their first and last
    # active day where this stream had NO raw data - a gap in collection.
    gap_lookup = {}
    for participant_id, group in active.groupby("Participant Id"):
        observed_days = pd.to_datetime(group["Date"].unique())
        full_range = pd.date_range(observed_days.min(), observed_days.max(), freq="D")
        gap_lookup[participant_id] = set(full_range.difference(observed_days))

    report = report.merge(
        first_last, left_on="Participant Id", right_index=True, how="left"
    )
    report_dates = pd.to_datetime(report["Date"])

    report["First/Last Day?"] = (
        (report_dates == report["First Day"])
        | (report_dates == report["Last Day"])
    )

    def is_next_to_gap(participant_id, date):
        gaps = gap_lookup.get(participant_id, set())
        return (
            (date - pd.Timedelta(days=1)) in gaps
            or (date + pd.Timedelta(days=1)) in gaps
        )

    report["Next to Gap?"] = [
        is_next_to_gap(pid, d)
        for pid, d in zip(report["Participant Id"], report_dates)
    ]

    return report.drop(columns=["First Day", "Last Day"])

## <a id="check-15-raw-data-vs-summary-metrics"></a>Check 15: Raw Data vs. Summary Metrics

For each data stream (GPS, Accelerometer, Calls + Texts), this compares two things for every participant-day: 1. Is there raw data present? 2. Are the summary metrics for that stream all filled in? Calls and Texts are checked together (not as two separate streams) because Willow runs if either one has raw data that day and fills in real zeros for whichever stream was quiet.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
stream_reports = []
forest_status_reports = []
forest_problem_report = pd.DataFrame(columns=["Stream", "Issue", "Forest Status"])

for stream_name, config in STREAMS.items():

    # Check that all required columns exist before running the validation
    required_stream_cols = [
        *config["raw_cols"],
        *config["metric_cols"]
    ]

    missing_stream_cols = [
        col for col in required_stream_cols
        if col not in df.columns
    ]

    if missing_stream_cols:
        print(
            f"SKIP: {stream_name} check skipped because "
            f"required column(s) are missing: {missing_stream_cols}"
        )
        continue

    # Run the validation
    stream_report = check_stream(
        df,
        stream_name,
        config["raw_cols"],
        config["metric_cols"]
    )

    if stream_report.empty:
        continue

    stream_report["Check"] = "Raw Data vs. Summary Metrics"
    stream_report["Column"] = pd.NA

    # Only days Forest actually processed can be judged; everything else is
    # reported separately below.
    stream_report = add_forest_status(stream_report, stream_name)
    forest_status_reports.append(stream_report[stream_report["Forest Status"] != "Processed"])
    stream_report = stream_report[stream_report["Forest Status"] == "Processed"]

    if stream_report.empty:
        continue

    # This adds whether each flagged day sits at the edge
    # of collection or next to a gap.
    stream_report = add_edge_context(df, stream_report, config["raw_cols"])

    stream_report = stream_report[
        [
            "Participant Id",
            "Date",
            "Check",
            "Stream",
            "Column",
            "Issue",
            "First/Last Day?",
            "Next to Gap?",
            "Forest Status",
            "Forest Run Started"
        ]
    ]

    stream_reports.append(stream_report)

print("===== Forest Processing Status of Flagged Rows =====")

not_processed = (
    pd.concat(forest_status_reports, ignore_index=True)
    if forest_status_reports
    else pd.DataFrame(columns=["Stream", "Issue", "Forest Status"])
)

if not_processed.empty:
    print("PASS: Every flagged row falls on a day Forest processed successfully.")
else:
    display(pd.crosstab(not_processed["Stream"], not_processed["Forest Status"]))

    never_processed = not_processed["Forest Status"] == "Never processed"
    print(
        f"SKIPPED: {never_processed.sum()} flagged row(s) fall on days Forest "
        "never ran on, so they are not counted as mismatches."
    )

    forest_problem_report = not_processed[~never_processed].copy()
    if not forest_problem_report.empty:
        forest_problem_report["Check"] = "Forest Processing Status"
        print(
            f"REPORTED: {len(forest_problem_report)} flagged row(s) where Forest "
            "failed or ran without writing any output."
        )
        display(
            forest_problem_report[
                ["Participant Id", "Date", "Stream", "Issue", "Forest Status", "Forest Error"]
            ]
        )

print("\n===== Raw Data vs. Summary Metrics Check (days Forest processed) =====")

if stream_reports:
    raw_metrics_report = pd.concat(
        stream_reports,
        ignore_index=True
    )

    # Break the total down by issue type, since "ALL metrics missing" is the
    # genuine processing gap we care about, while "SOME metrics missing" is
    # often expected (e.g. stddev needs 2+ flights) and lower priority.
    issue_counts = raw_metrics_report["Issue"].value_counts()

    print(
        f"FAIL: {len(raw_metrics_report)} "
        "raw-versus-metric mismatch(es) found."
    )
    for issue_label, count in issue_counts.items():
        print(f"  {count} — {issue_label}")

    display(raw_metrics_report)

else:
    raw_metrics_report = pd.DataFrame(
        columns=[
            "Participant Id", "Date", "Check", "Stream", "Column", "Issue",
            "First/Last Day?", "Next to Gap?", "Forest Status", "Forest Run Started"
        ]
    )
    print("PASS: No raw-versus-metric mismatches found.")

# Both reports are added to all_reports at the end of the GPS drill-down, after the
# upload timing check below has removed any late-upload rows.

## <a id="check-16-raw-data-uploaded-too-late"></a>Check 16: Raw Data Uploaded Too Late?

A "raw data present but ALL metrics missing" row can just mean the raw data reached the server after Forest's last run on that day, so Forest never saw it. For those rows, this compares when the day's raw files were uploaded (the `get-participant-upload-history/v1` endpoint) against when the most recent successful Forest run covering that day started. Rows whose raw data all arrived after that run are not mismatches; they're removed and counted separately.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
import json

# >>> CHANGE THIS if your study is not on US Eastern time. <<<
# WHY: raw file times are stored in UTC. We convert them to the study's local
# time so a file lands on the same calendar day that Forest assigned it to.
STUDY_TIMEZONE = "America/New_York"  # study timezone, used to bucket raw files into days
UPLOAD_CACHE_DIR = "upload_history_cache"

# Upload file names start with the stream folder, e.g. "gps/1763654506883.csv".
UPLOAD_STREAM_PREFIXES = {
    "GPS": ["gps"],
    "Accelerometer": ["accel"],
    "Calls+Texts": ["callLog", "textsLog"],
}


def load_upload_history(participant_id):
    os.makedirs(UPLOAD_CACHE_DIR, exist_ok=True)
    path = os.path.join(UPLOAD_CACHE_DIR, f"{participant_id}.json")

    if not os.path.exists(path):
        response = requests.post(
            f"{BEIWE_SERVER}/get-participant-upload-history/v1",
            data={**BEIWE_CREDENTIALS, "participant_id": participant_id, "omit_keys": "true"},
        )
        response.raise_for_status()
        with open(path, "wb") as f:
            f.write(response.content)

    with open(path, "rb") as f:
        uploads = pd.DataFrame(json.loads(f.read()), columns=["File Size", "Uploaded", "File Name"])

    uploads["Stream Prefix"] = uploads["File Name"].str.split("/").str[0]
    # The 13-digit number is when the phone created the file. Some names carry a
    # "-duplicate-..." suffix after ".csv", so the pattern isn't anchored to the end.
    created_ms = pd.to_numeric(uploads["File Name"].str.extract(r"/(\d{13})\.csv")[0], errors="coerce")
    uploads["Created Day"] = (
        pd.to_datetime(created_ms, unit="ms", utc=True)
        .dt.tz_convert(STUDY_TIMEZONE)
        .dt.tz_localize(None)
        .dt.normalize()
    )
    uploads["Uploaded"] = pd.to_datetime(uploads["Uploaded"], utc=True)
    return uploads


# WHY: Forest can only use data that has already reached the server when it runs.
# If a phone uploaded a day's data AFTER Forest ran, the missing metrics are a
# timing issue (Forest needs to re-run), not a Forest bug.
print("===== Raw Upload Timing Check =====")

timing_mask = raw_metrics_report["Issue"] == "Raw data present but ALL metrics missing"
raw_metrics_report["Upload Timing"] = pd.NA

if not timing_mask.any():
    print("PASS: No 'ALL metrics missing' rows to check.")
else:
    upload_histories = {}
    timing_labels = []

    for _, row in raw_metrics_report.loc[timing_mask].iterrows():
        pid = row["Participant Id"]
        if pid not in upload_histories:
            upload_histories[pid] = load_upload_history(pid)
        uploads = upload_histories[pid]

        day = pd.to_datetime(row["Date"])
        # A file keeps collecting after it's created, so data for this day can sit in a
        # file the phone started the day before.
        day_files = uploads[
            uploads["Created Day"].between(day - pd.Timedelta(days=1), day)
            & uploads["Stream Prefix"].isin(UPLOAD_STREAM_PREFIXES[row["Stream"]])
        ]

        run_started = pd.to_datetime(row["Forest Run Started"], utc=True)
        total_bytes = day_files["File Size"].sum()
        late_bytes = day_files.loc[day_files["Uploaded"] > run_started, "File Size"].sum()

        if total_bytes == 0:
            timing_labels.append("No upload records found")
        elif late_bytes == total_bytes:
            timing_labels.append("All raw uploaded after Forest ran")
        elif late_bytes > 0:
            timing_labels.append("Some raw uploaded after Forest ran")
        else:
            timing_labels.append("Raw uploaded before Forest ran")

    raw_metrics_report.loc[timing_mask, "Upload Timing"] = timing_labels

    print(raw_metrics_report.loc[timing_mask, "Upload Timing"].value_counts().to_string())

    late_mask = raw_metrics_report["Upload Timing"] == "All raw uploaded after Forest ran"
    if late_mask.any():
        late_upload_report = raw_metrics_report[late_mask]
        raw_metrics_report = raw_metrics_report[~late_mask].reset_index(drop=True)
        print(
            f"\nREMOVED: {late_mask.sum()} row(s) whose raw data all arrived after Forest's "
            "last run, so they are not mismatches."
        )
        display(late_upload_report[["Participant Id", "Date", "Stream", "Forest Run Started"]])

## <a id="check-17-gps-missing-metrics-drill-down"></a>Check 17: GPS Missing-Metrics Drill-Down

Tests each GPS row where raw data is present but every metric is missing against Jasmine's actual rules, using the raw GPS files for those participants (the `get-data/v1` endpoint; only row counts are cached, never coordinates). This covers rows from both the Forest Processing Status and Raw Data vs. Summary Metrics checks. Two rows get an **Error Code**: (1) **Insufficient GPS Data** - on a run that wrote no output, 5% or fewer of the GPS files in that run's data window had more than 60 rows, so Jasmine's `gps_quality_check` skipped the participant. (2) **Low GPS Data Accuracy** - Jasmine's quality check passed, but every GPS point that day has accuracy of 51 m or worse, which Jasmine drops, so no trajectory exists for that day. Usually a location-permission setting on the phone. Remaining rows have usable points and get a **Likely Cause** instead: Jasmine builds daily windows in fixed 24-hour steps from the first day's midnight, so across a daylight-saving change the last day of data gets no row, or a row holds the next day's data.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
import io
import zipfile

print("===== GPS 'ALL Metrics Missing' Drill-Down =====")

# WHY these exact numbers? They are copied from Jasmine's (Forest's GPS tree) own
# default settings, so this check uses exactly the same rules Forest used when it
# decided to skip a participant or throw out GPS points.
# Jasmine's defaults: gps_quality_check and Hyperparameters.quality_threshold / accuracylim
GPS_FILE_MIN_ROWS = 60  # a file only counts toward quality with MORE than this many rows
GPS_QUALITY_THRESHOLD = 0.05  # a participant is skipped at or below this share of files
GPS_ACCURACY_LIMIT = 51  # meters; points at or above this are dropped

INSUFFICIENT_GPS_DATA = "Insufficient GPS Data"
LOW_GPS_DATA_ACCURACY = "Low GPS Data Accuracy"

GPS_SUMMARY_CACHE_DIR = "gps_summary_cache"


# Raw GPS is only needed for row counts, so coordinates are never written to disk.
def load_gps_summary(participant_id):
    os.makedirs(GPS_SUMMARY_CACHE_DIR, exist_ok=True)
    files_path = os.path.join(GPS_SUMMARY_CACHE_DIR, f"{participant_id}_files.csv")
    days_path = os.path.join(GPS_SUMMARY_CACHE_DIR, f"{participant_id}_days.csv")

    if not (os.path.exists(files_path) and os.path.exists(days_path)):
        response = requests.post(
            f"{BEIWE_SERVER}/get-data/v1",
            data={
                **BEIWE_CREDENTIALS,
                "study_id": validation_study_id,
                "participant_ids": json.dumps([participant_id]),
                "data_streams": json.dumps(["gps"]),
            },
            timeout=1800,
        )
        response.raise_for_status()

        file_rows = []
        points = []
        with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
            for name in archive.namelist():
                if "/gps/" not in name or not name.endswith(".csv"):
                    continue
                gps = pd.read_csv(archive.open(name), usecols=["timestamp", "accuracy"])
                # File names are the hour they cover, e.g. "2025-08-27 18_00_00+00_00.csv".
                file_rows.append((os.path.basename(name)[:19].replace("_", ":"), len(gps)))
                points.append(gps)

        points = pd.concat(points, ignore_index=True)
        points["Day"] = (
            pd.to_datetime(points["timestamp"], unit="ms", utc=True)
            .dt.tz_convert(STUDY_TIMEZONE)
            .dt.tz_localize(None)
            .dt.normalize()
        )
        points["Usable"] = points["accuracy"] < GPS_ACCURACY_LIMIT

        pd.DataFrame(file_rows, columns=["File Hour", "Rows"]).to_csv(files_path, index=False)
        (
            points.groupby("Day")
            .agg(Points=("Usable", "size"), Usable_Points=("Usable", "sum"))
            .reset_index()
            .to_csv(days_path, index=False)
        )

    files = pd.read_csv(files_path)
    files["File Hour"] = pd.to_datetime(files["File Hour"], utc=True)
    days = pd.read_csv(days_path, parse_dates=["Day"]).set_index("Day")
    return files, days


# Same calculation as Jasmine's gps_quality_check, on the files in one run's data window.
def gps_quality_share(files, data_start, data_end):
    window_start = pd.Timestamp(data_start, tz=STUDY_TIMEZONE)
    window_end = pd.Timestamp(data_end, tz=STUDY_TIMEZONE) + pd.Timedelta(days=1)
    in_window = files[(files["File Hour"] >= window_start) & (files["File Hour"] < window_end)]
    return (in_window["Rows"] > GPS_FILE_MIN_ROWS).sum() / (len(in_window) + 0.0001)


def classify_gps_row(row):
    files, days = gps_summaries[row["Participant Id"]]

    quality_share = pd.NA
    if row["Forest Status"] == "Ran, no output":
        quality_share = gps_quality_share(files, row["Forest Data Start"], row["Forest Data End"])

    day = pd.Timestamp(row["Date"])
    points = days.loc[day, "Points"] if day in days.index else 0
    usable_points = days.loc[day, "Usable_Points"] if day in days.index else 0

    if pd.notna(quality_share) and quality_share <= GPS_QUALITY_THRESHOLD:
        error_code = INSUFFICIENT_GPS_DATA
    elif points > 0 and usable_points == 0:
        error_code = LOW_GPS_DATA_ACCURACY
    else:
        error_code = pd.NA

    return pd.Series(
        {
            "Error Code": error_code,
            "Share of Files Over 60 Rows": quality_share,
            "GPS Points": points,
            "Usable GPS Points": usable_points,
        }
    )


# Jasmine steps through days in fixed 24-hour blocks from the first day's midnight, so when a
# participant's GPS starts and ends on opposite sides of a clock change, their last day gets no row.
def utc_offset(day):
    return pd.Timestamp(day).tz_localize(STUDY_TIMEZONE).utcoffset()


def gps_dst_cause(row):
    first_gps_day, last_gps_day = gps_span.loc[row["Participant Id"], ["min", "max"]]
    if pd.Timestamp(row["Date"]) == last_gps_day and utc_offset(first_gps_day) != utc_offset(last_gps_day):
        return "Last GPS day across a clock change (Forest: Jasmine's DST day-window bug drops this day)"
    return "Usable GPS points (Forest: likely Jasmine's DST day-window bug, which drops or shifts a day)"


def gps_all_missing(report):
    return (
        (report["Stream"] == "GPS")
        & (report["Issue"] == "Raw data present but ALL metrics missing")
        & (report["Forest Status"] != "Failed")
    )


raw_metrics_report["Error Code"] = pd.NA
raw_metrics_report["Likely Cause"] = pd.NA
forest_problem_report["Error Code"] = pd.NA

gps_participants = pd.concat(
    [report.loc[gps_all_missing(report), "Participant Id"] for report in [forest_problem_report, raw_metrics_report]]
).unique()

if "GPS" not in STREAMS or len(gps_participants) == 0:
    print("PASS: No GPS 'ALL metrics missing' rows to classify.")
else:
    gps_summaries = {pid: load_gps_summary(pid) for pid in gps_participants}

    gps_details = []
    for report in [forest_problem_report, raw_metrics_report]:
        mask = gps_all_missing(report)
        if mask.any():
            detail = report[mask].apply(classify_gps_row, axis=1)
            report.loc[mask, "Error Code"] = detail["Error Code"]
            gps_details.append(report.loc[mask, ["Participant Id", "Date", "Forest Status"]].join(detail))
    gps_detail = pd.concat(gps_details, ignore_index=True)

    gps_raw_col = STREAMS["GPS"]["raw_cols"][0]
    gps_present = df[df[gps_raw_col].fillna(0) > 0]
    gps_span = pd.to_datetime(gps_present["Date"]).groupby(gps_present["Participant Id"]).agg(["min", "max"])

    needs_cause = gps_all_missing(raw_metrics_report) & raw_metrics_report["Error Code"].isna()
    if needs_cause.any():
        raw_metrics_report.loc[needs_cause, "Likely Cause"] = raw_metrics_report[needs_cause].apply(
            gps_dst_cause, axis=1
        )

    print(f"Classified {len(gps_detail)} GPS 'ALL metrics missing' row(s):")
    print(gps_detail["Error Code"].value_counts(dropna=False).to_string())
    display(gps_detail)

all_reports.append(forest_problem_report)
all_reports.append(raw_metrics_report)


## <a id="final-step-save-the-report"></a>Final Step: Save the Report

This saves the final report as a CSV with a complete table of any issues found.

[↑ Back to Table of Contents](#table-of-contents)

In [ ]:
report_columns = [
    "Participant Id",
    "Date",
    "Check",
    "Stream",
    "Column",
    "Issue",
    "Error Code",
    "Forest Status",
    "Forest Error",
    "Upload Timing",
    "First/Last Day?",
    "Next to Gap?",
    "Likely Cause"
]

if all_reports:
    final_report = pd.concat(
        all_reports,
        ignore_index=True
    )

    for col in report_columns:
        if col not in final_report.columns:
            final_report[col] = pd.NA

    final_report = final_report[report_columns]

    # Guard against duplicate rows from re-running earlier check cells
    # (e.g. re-executing a check cell without restarting the kernel first,
    # which would otherwise append the same issues to all_reports twice)
    before_dedup = len(final_report)
    final_report = final_report.drop_duplicates().reset_index(drop=True)
    duplicates_removed = before_dedup - len(final_report)

else:
    final_report = pd.DataFrame(
        columns=report_columns
    )
    duplicates_removed = 0

print("===== Complete Validation Report =====")
print(f"Total issues found: {len(final_report)}")

if duplicates_removed > 0:
    print(
        f"NOTE: Removed {duplicates_removed} duplicate row(s) — "
        "likely from a check cell being run more than once. "
        "Consider Kernel > Restart & Run All for a clean run."
    )

if final_report.empty:
    print("PASS: No validation issues found.")
else:
    display(final_report)

final_report.to_csv(
    "validation_report.csv",
    index=False
)

print("Saved validation_report.csv")